In [15]:
# ================================================================
#   WEATHER DASHBOARD (TEMP + WIND + CALENDAR + GRAPH)
# ================================================================

!pip install requests ipywidgets matplotlib --quiet

import requests
import datetime
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt

# ================================================================
#   WEATHER ICONS
# ================================================================

def get_icon(code):
    if code == 0:        return "☀️"
    elif code <= 2:      return "🌤️"
    elif code <= 3:      return "☁️"
    elif code <= 49:     return "🌫️"
    elif code <= 59:     return "🌦️"
    elif code <= 69:     return "🌧️"
    elif code <= 79:     return "❄️"
    elif code <= 84:     return "🌨️"
    elif code <= 99:     return "⛈️"
    else:                return "🌡️"

# ================================================================
#   GEOCODING
# ================================================================

def geocode(city):
    r = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1}
    )
    data = r.json()

    if "results" not in data:
        return None

    loc = data["results"][0]
    return {
        "label": f"{loc['name']}, {loc.get('country','')}",
        "lat": loc["latitude"],
        "lon": loc["longitude"]
    }

# ================================================================
#   FETCH WEATHER DATA (TEMP + RAIN + WIND)
# ================================================================

def fetch_weather(lat, lon):
    year = datetime.datetime.now().year - 1

    try:
        r = requests.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params={
                "latitude": lat,
                "longitude": lon,
                "start_date": f"{year}-01-01",
                "end_date": f"{year}-12-31",
                "daily": "temperature_2m_max,temperature_2m_min,weathercode,precipitation_sum,windspeed_10m_max",
                "timezone": "auto"
            }
        )

        data = r.json()
        if "daily" not in data:
            return {}

        daily = data["daily"]
        result = {}

        for i, date in enumerate(daily["time"]):
            result[date] = {
                "hi": round(daily["temperature_2m_max"][i]),
                "lo": round(daily["temperature_2m_min"][i]),
                "code": daily["weathercode"][i],
                "rain": daily["precipitation_sum"][i] or 0,
                "wind": round(daily["windspeed_10m_max"][i])
            }

        return result

    except Exception as e:
        print("Error:", e)
        return {}

# ================================================================
#   CALENDAR UI
# ================================================================

MONTHS = ["January","February","March","April","May","June",
          "July","August","September","October","November","December"]

def build_calendar_html(data, city, month):

    year = datetime.datetime.now().year - 1
    days = (datetime.date(year, month % 12 + 1, 1) - datetime.timedelta(days=1)).day if month < 12 else 31

    html_days = ""

    for d in range(1, days + 1):
        ds = f"{year}-{str(month).zfill(2)}-{str(d).zfill(2)}"
        v = data.get(ds, {})

        icon = get_icon(v["code"]) if v else "—"
        hi   = f"{v['hi']}°" if v else "—"
        lo   = f"{v['lo']}°" if v else "—"
        wind = f"{v['wind']} km/h" if v else "—"

        html_days += f"""
        <div style="
            background: rgba(255,255,255,0.08);
            border-radius: 12px;
            padding: 10px;
            text-align: center;
            backdrop-filter: blur(10px);
        ">
            <div style="font-size:12px;color:#bbb;">{d}</div>
            <div style="font-size:26px;">{icon}</div>
            <div style="font-weight:bold;">{hi}</div>
            <div style="font-size:12px;color:#aaa;">{lo}</div>
            <div style="font-size:11px;color:#8ecae6;">🌬 {wind}</div>
        </div>
        """

    html = f"""
    <div style="
        background: linear-gradient(135deg,#0f2027,#203a43,#2c5364);
        padding:20px;
        border-radius:20px;
        color:white;
        font-family:Segoe UI;
        max-width:900px;
    ">

        <h2>📍 {city}</h2>
        <h3>{MONTHS[month-1]}</h3>

        <div style="
            display:grid;
            grid-template-columns: repeat(7,1fr);
            gap:10px;
            margin-top:15px;
        ">
            {html_days}
        </div>

    </div>
    """

    return html

# ================================================================
#   TEMPERATURE GRAPH
# ================================================================

def show_temp_graph(data, city, month):
    year = datetime.datetime.now().year - 1

    days = []
    hi = []
    lo = []

    for d in range(1, 32):
        ds = f"{year}-{str(month).zfill(2)}-{str(d).zfill(2)}"
        if ds in data:
            days.append(d)
            hi.append(data[ds]["hi"])
            lo.append(data[ds]["lo"])

    plt.figure()
    plt.plot(days, hi, label="Max Temp")
    plt.plot(days, lo, label="Min Temp")
    plt.title(f"Temperature Graph - {city}")
    plt.xlabel("Day")
    plt.ylabel("Temperature (°C)")
    plt.legend()
    plt.show()

# ================================================================
#   WIND GRAPH
# ================================================================

def show_wind_graph(data, city, month):
    year = datetime.datetime.now().year - 1

    days = []
    wind = []

    for d in range(1, 32):
        ds = f"{year}-{str(month).zfill(2)}-{str(d).zfill(2)}"
        if ds in data:
            days.append(d)
            wind.append(data[ds]["wind"])

    plt.figure()
    plt.plot(days, wind, label="Wind Speed")
    plt.title(f"Wind Speed Graph - {city}")
    plt.xlabel("Day")
    plt.ylabel("Wind Speed (km/h)")
    plt.legend()
    plt.show()

# ================================================================
#   MAIN APP
# ================================================================

city_box = widgets.Text(value="Delhi", description="City:")

month_box = widgets.Dropdown(
    options=[(m, i+1) for i, m in enumerate(MONTHS)],
    value=datetime.datetime.now().month,
    description="Month:"
)

view_toggle = widgets.ToggleButtons(
    options=["Calendar", "Temp Graph", "Wind Graph"],
    description="View:"
)

btn = widgets.Button(description="Get Weather 🌤")
out = widgets.Output()

display(widgets.HBox([city_box, month_box, view_toggle, btn]), out)

cache = {}

def on_click(b):
    with out:
        clear_output()

        city = city_box.value.strip()
        if not city:
            print("Enter city name")
            return

        loc = geocode(city)
        if not loc:
            print("City not found")
            return

        if city not in cache:
            print("Fetching weather data...")
            data = fetch_weather(loc["lat"], loc["lon"])
            cache[city] = data

        if view_toggle.value == "Calendar":
            html = build_calendar_html(cache[city], loc["label"], month_box.value)
            display(HTML(html))

        elif view_toggle.value == "Temp Graph":
            show_temp_graph(cache[city], loc["label"], month_box.value)

        else:
            show_wind_graph(cache[city], loc["label"], month_box.value)

btn.on_click(on_click)

Output()